# LRRM

此示例演示如何使用 `skrf` 的 LRRM 校准。LRRM 代表 Line-Reflect-Reflect-Match，这些是校准所需的校准标准。LRRM 有几种不同的实现，使用了略微不同的假设和匹配模型。`skrf` 的 LRRM 校准使用以下假设：

 * 线路标准需要精确知道。
 * 第一个反射的相位需要在 90 度以内。它可以是损耗性的，并且假设在两个端口上是相同的。
 * 第二个反射的 |S11| 假设为已知，其相位也需要在 90 度以内，并且假设在两个端口上是相同的。这两个反射需要不同，并且它们的相位差应为 180 度以获得最佳精度。
 * 匹配标准假设为一个与未知电感串联的已知电阻。匹配只需要在第一个端口上测量。

校准标准和测量数据需要按照上述顺序提供给校准例程。如果遵循上述假设，校准可以精确求解反射、匹配和校准参数。

## 使用合成数据的 LRRM 示例

### 设置

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

import skrf
from skrf.calibration import LRRM, TwelveTerm, terminate
from skrf.media import Coaxial
from skrf.network import two_port_reflect

skrf.stylely()

### 生成示例数据

我们首先生成一些合成误差盒和校准标准。我们将有两套校准标准。用于校准的真实标准具有寄生效应，而没有寄生效应的近似标准将提供给校准算法。

In [ ]:
freq = skrf.Frequency(1, 100, 100, 'GHz')

# 1.0 mm coaxial media for calibration error boxes
coax = Coaxial(freq, z0_port=50, Dint=0.44e-3, Dout=1.0e-3, sigma=1e8)

# Generate random error boxes
X = coax.random(n_ports=2, name='X')
Y = coax.random(n_ports=2, name='Y')

# Random switch terms
gamma_f = coax.random(n_ports=1, name='gamma_f')
gamma_r = coax.random(n_ports=1, name='gamma_r')

# Our guess of the standards. We assume they don't have any parasitics.
oo_i = coax.open(nports=2, name='open')
ss_i = coax.short(nports=2, name='short')
# Match is only measured on one port. Resistance can be different from 50 ohms.
m_i = coax.resistor(R=50, name='r') ** coax.short(nports=1)
# Thru must be known exactly
thru = coax.line(d=100, unit='um', name='thru')

# Actual reflects with parasitics. They must be identical in both ports.
# Short is slightly lossy.
ss = coax.line(d=200, unit='um') ** coax.load(-0.98,nports=2, name='short') ** coax.line(d=200, unit='um')
oo = coax.shunt_capacitor(10e-15) ** coax.open(nports=2, name='open') ** coax.shunt_capacitor(10e-15)

# Match standard has inductance in series.
match_l = 40e-12
l = coax.inductor(L=match_l)
m = l**m_i

# Make two-port of the match with open on the second port.
mm = two_port_reflect(m, coax.open(nports=1))
# Make two-port for the match standard
mm_i = coax.match(nports=2, name='load')

# These are our guesses of the calibration standards.
approx_ideals = [
    thru,
    ss_i,
    oo_i,
    mm_i
    ]

# These are the actual standards with parasitics.
ideals = [
    thru,
    ss,
    oo,
    mm
    ]

# Make measurement of the standards using the random error boxes and switch terms.
measured = [terminate(X**k**Y, gamma_f, gamma_r) for k in ideals]

### 可视化标准

In [ ]:
plt.figure()
oo.plot_s_smith(m=0, n=0, label='Open')
ss.plot_s_smith(m=0, n=0, label='Short')
mm.plot_s_smith(m=0, n=0, label='Match')

plt.figure()
oo.plot_s_db(m=0, n=0, label='Open')
ss.plot_s_db(m=0, n=0, label='Short')
mm.plot_s_db(m=0, n=0, label='Match')

### LRRM 校准

我们假装不知道具有寄生效应的实际标准，并且只给出没有寄生效应的标准近似值和具有寄生效应的标准的测量值。

In [ ]:
cal = LRRM(
    ideals = approx_ideals,
    measured = measured,
    switch_terms = [gamma_f, gamma_r])

### 可视化解出的标准

LRRM 校准求解真实标准。我们可以从 `cal` 对象获取解出的标准。解出的标准应该与上面的实际标准匹配。

In [ ]:
plt.figure()
cal.solved_r2.plot_s_smith(m=0, n=0, label='Open')
cal.solved_r1.plot_s_smith(m=0, n=0, label='Short')
cal.solved_m.plot_s_smith(m=0, n=0, label='Match')

plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Open')
cal.solved_r1.plot_s_db(m=0, n=0, label='Short')
cal.solved_m.plot_s_db(m=0, n=0, label='Match')

匹配的解出电感也作为校准输出给出。它以数组形式给出，但使用默认选项时，在所有频率上拟合单个电感。

In [ ]:
solved_match_l = cal.solved_l[0]
print(f'Solved inductance {1e12*solved_match_l:.1f} pH, actual inductance {1e12*match_l:.1f} pH')

### 校准 DUT

可以使用 `apply_cal` 方法对测量的 DUT 进行校准。S 参数应该完全匹配。

In [ ]:
# Let's generate a DUT: 5 mm long 75 ohm line.
dut = coax.line(d=5, unit='mm', z0=75)

dut_measured = terminate(X**dut**Y, gamma_f, gamma_r)
dut_cal = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

### 使用反射 |S11| 进行校准验证

在校准过程中，第二个反射 |S11| 被假定为已知（在这种情况下 |S11| = 1），但是当将单个电感拟合到匹配标准时，此假设可能会被打破。如果真实匹配没有很好地建模为与电感串联的已知电阻，它会导致反射标准的无损耗性被违反。通过绘制反射的绝对值，我们可以了解校准假设的有多好。

让我们首先绘制之前校准中的开路 |S11|。如果一切正常，它应该正好是 0 dB。

In [ ]:
plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Solved open')
plt.ylim([-0.01, 0.01])

### 使用容性匹配的校准

匹配的 LRRM 校准模型是电阻与电感串联。如果匹配还具有并联电容，它将不能被正确求解，并且校正后的测量会出现错误。

让我们定义一个新的匹配标准并使用它重新进行校准。

### 可视化容性匹配和已解出的匹配

校准试图尽可能好地拟合电感到匹配，但它无法用单个电感精确建模匹配。最接近的拟合是负值电感。

In [ ]:
plt.figure()
mm.plot_s_smith(m=0, n=0, label='Actual')
cal.solved_m.plot_s_smith(m=0, n=0, label='Solved')

plt.figure()
mm.plot_s_db(m=0, n=0, label='Actual')
cal.solved_m.plot_s_db(m=0, n=0, label='Solved')

绘制开路的 |S11| 揭示了解出的开路不是无损耗的，这表明一些校准假设被违反了。

Plotting the |S11| of the open reveals that the solved open is not lossless indicating that some of the calibration assumptions were violated.

In [ ]:
plt.figure()
cal.solved_r2.plot_s_db(m=0, n=0, label='Solved open')

### 使用不正确建模的匹配进行 DUT 测量

不正确的匹配导致校准参数中出现错误。误差在高频下增加，此时匹配建模误差更大。

In [ ]:
dut_cal2 = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal2.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal2.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

### 使用电感和电容的匹配拟合

In [ ]:
# Redo the calibration using LC match model
cal = LRRM(
    ideals = approx_ideals,
    measured = measured,
    match_fit = 'lc',
    switch_terms = [gamma_f, gamma_r]
    )

In [ ]:
plt.figure()
mm.plot_s_smith(m=0, n=0, label='Actual')
cal.solved_m.plot_s_smith(m=0, n=0, label='Solved')

plt.figure()
mm.plot_s_db(m=0, n=0, label='Actual')
cal.solved_m.plot_s_db(m=0, n=0, label='Solved')

In [ ]:
solved_match_l = cal.solved_l[0]
solved_match_c = cal.solved_c[0]
print(f'Solved inductance {1e12*solved_match_l:.1f} pH, actual inductance {1e12*match_l:.1f} pH')
print(f'Solved capacitance {1e15*solved_match_c:.1f} fF, actual capacitance {1e15*match_c:.1f} fF')

应用校正现在到测量应该给出接近的拟合。

In [ ]:
dut_cal3 = cal.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal3.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal3.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])

## 与 SOLT 校准的比较

传统的双端口 SOLT 校准假设所有标准都精确知道。我们可以使用相同的测量和近似已知标准来比较它的性能。对于 SOLT，匹配还需要在第二个端口上测量，我们假设它与第一个端口相同。随机生成的误差盒使校准特别困难。

In [ ]:
# SOLT requires match measurement on both ports.
mm = two_port_reflect(m, m)
measured[3] = terminate(X**mm**Y, gamma_f, gamma_r)

# TwelveTerm assumes that thru is last.
cal12 = TwelveTerm(
    ideals = list(reversed(approx_ideals)),
    measured = list(reversed(measured)),
    n_thrus = 1,
    )

dut_cal12 = cal12.apply_cal(dut_measured)

plt.figure()
dut.plot_s_db(m=0, n=0, label='Actual S11')
dut.plot_s_db(m=1, n=0, label='Actual S21')
dut_cal12.plot_s_db(m=0, n=0, label='Calibrated S11')
dut_cal12.plot_s_db(m=1, n=0, label='Calibrated S21')
plt.ylim([-20, 5])